In [ ]:
%pip install mysql-connector-python pandas



SyntaxError: invalid syntax (581819536.py, line 2)

In [ ]:
import mysql.connector
from mysql.connector import errorcode
import pandas as pd
import ast

# --- CONFIGURATION ---
config = {
  'user': 'YOUR_DB_USER',
  'password': 'YOUR_PASSWORD',
  'host': 'YOUR_DB_HOST',
  'database': 'YOUR_DB_NAME',
  'raise_on_warnings': True
}

# 1. SETUP DATABASE CONNECTION
def get_connection():
    try:
        cnx = mysql.connector.connect(**config)
        print(f"Successfully connected to {config['database']} at {config['host']}")
        return cnx
    except mysql.connector.Error as err:
        if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print("Something is wrong with your user name or password")
        elif err.errno == errorcode.ER_BAD_DB_ERROR:
            print("Database does not exist")
        else:
            print(err)
        return None

# Test connection
cnx = get_connection()
if cnx:
    cnx.close()

In [ ]:
# 2. LOAD AND PREP DATA
csv_file = 'anime_dataset.csv'
print("Loading CSV...")
# Using latin-1 as seen in previous context, just in case
df = pd.read_csv(csv_file, encoding='latin-1')

# Convert string representations of lists "['Action']" to actual lists ['Action']
df['genres'] = df['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
df['studios'] = df['studios'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

print(f"Loaded {len(df)} rows.")
print("Sample data:")
print(df.head(2))

Loading CSV...
Loaded 1050 rows.
Sample data:
             title  score                                             genres  \
0  Attack on Titan   8.57  [Action, Award Winning, Drama, Suspense, Gore,...   
1       Death Note   8.62   [Supernatural, Suspense, Psychological, Shounen]   

   episodes  popularity  members       studios    year  
0      25.0           1  4245518  [Wit Studio]  2013.0  
1      37.0           2  4186098    [Madhouse]  2006.0  


In [ ]:
# 3. PROCESSING FUNCTION
def process_data(row, cursor):
    # A. Insert into 'animes' table
    # Note: anime_id is AUTO_INCREMENT, so we don't insert it.
    add_anime = ("INSERT INTO animes "
                 "(title, score, episodes, popularity, members, year) "
                 "VALUES (%s, %s, %s, %s, %s, %s)")

    data_anime = (row['title'], row['score'], row['episodes'], row['popularity'], row['members'], row['year'])

    cursor.execute(add_anime, data_anime)
    anime_id = cursor.lastrowid

    # B. Handle Genres
    for genre_name in row['genres']:
        # Check/Insert Genre using 'genre_name' and 'genre_id'
        cursor.execute("SELECT genre_id FROM genres WHERE genre_name = %s", (genre_name,))
        result = cursor.fetchone()

        if result:
            genre_id = result[0]
        else:
            cursor.execute("INSERT INTO genres (genre_name) VALUES (%s)", (genre_name,))
            genre_id = cursor.lastrowid

        # Link in junction table
        cursor.execute("INSERT IGNORE INTO anime_genres (anime_id, genre_id) VALUES (%s, %s)", (anime_id, genre_id))

    # C. Handle Studios
    for studio_name in row['studios']:
        # Check/Insert Studio using 'studio_name' and 'studio_id'
        cursor.execute("SELECT studio_id FROM studios WHERE studio_name = %s", (studio_name,))
        result = cursor.fetchone()

        if result:
            studio_id = result[0]
        else:
            cursor.execute("INSERT INTO studios (studio_name) VALUES (%s)", (studio_name,))
            studio_id = cursor.lastrowid

        # Link in junction table
        cursor.execute("INSERT IGNORE INTO anime_studios (anime_id, studio_id) VALUES (%s, %s)", (anime_id, studio_id))

# 4. EXECUTE
print("Starting import. This may take a moment...")
cnx = get_connection()
if cnx:
    cursor = cnx.cursor()
    try:
        for index, row in df.iterrows():
            process_data(row, cursor)
            # Optional: Print progress every 100 rows
            if index % 100 == 0:
                print(f"Processed {index} rows...")

        cnx.commit()
        print("Import successful! All data committed.")
    except Exception as e:
        cnx.rollback()
        print(f"Error occurred during import: {e}")
        print("Rolled back changes.")
    finally:
        cursor.close()
        cnx.close()

Starting import. This may take a moment...
Successfully connected to btmtestdb2 at 35.227.168.92
Processed 0 rows...
Processed 0 rows...
Processed 100 rows...
Processed 100 rows...
Processed 200 rows...
Processed 200 rows...
Processed 300 rows...
Processed 300 rows...
Processed 400 rows...
Processed 400 rows...
Processed 500 rows...
Processed 500 rows...
Processed 600 rows...
Processed 600 rows...
Processed 700 rows...
Processed 700 rows...
Processed 800 rows...
Processed 800 rows...
Processed 900 rows...
Processed 900 rows...
Processed 1000 rows...
Processed 1000 rows...
Import successful! All data committed.
Import successful! All data committed.
